# 00 — Catalogue audit

Freeze the live NASA Exoplanet Archive TOI table, apply the strict label rule, and record provenance before downloading any telescope data.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
from transit_hunter.catalog import freeze_catalogue

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
METADATA_DIR = PROJECT_ROOT / 'data' / 'metadata'
METADATA_DIR

In [ ]:
# This performs one live TAP request and writes the raw CSV, labels snapshot, KP file, and manifest.
# Commit the resulting metadata files to make this exact label source immutable.
manifest = freeze_catalogue(METADATA_DIR, retrieved_on=date.today())
pd.Series(manifest['audit'])

In [ ]:
labels = pd.read_csv(METADATA_DIR / 'labels_snapshot.csv')
display(labels['tfopwg_disp'].value_counts().rename('count'))
display(labels.groupby('label')['toi'].count().rename('count'))

assert labels['toi'].is_unique, 'A TOI may occur only once in a snapshot.'
# TICs can repeat when a star has several transit-like signals; they are grouped later.
tic_multiplicity = labels.groupby('tid')['toi'].nunique()
print(f'{len(labels):,} supervised TOIs across {labels.tid.nunique():,} TICs')
print(f'{(tic_multiplicity > 1).sum():,} TICs have multiple retained TOIs')

## Exit criteria

- The manifest names the retrieval date, endpoint, query, hash, and class counts.
- `labels_snapshot.csv` contains only CP, FP, and FA.
- A future temporal split keeps every TIC in one partition.